# 10 — Set up Lakebase (Postgres) for the Hire Right app

Creates a **Lakebase Provisioned** PostgreSQL instance and wires it up so the
Databricks App serves candidate & job data with low latency, and can persist
HR annotations.

This notebook is **idempotent**:
- If the Lakebase instance already exists it is **reused** (started if stopped) — creation is skipped.
- The database catalog, synced tables, and annotations table all use *create-if-missing* semantics.

What it builds:
1. Lakebase instance `${lakebase_instance_name}`.
2. A Postgres-backed **database catalog** that exposes synced tables in Unity Catalog.
3. **Synced tables** (Delta → Postgres) for the tables the app UI reads:
   `candidate_scoring_summary` and `job_requirements`.
4. A native, transactional **`candidate_annotations`** table for HR notes,
   joinable to candidate info on `candidate_id`.

In [ ]:
%pip install -q --upgrade databricks-sdk "psycopg[binary]"

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog",                 "bldemos",              "UC Catalog (Delta source)")
dbutils.widgets.text("schema",                  "hrd_2030",             "UC Schema")
dbutils.widgets.text("lakebase_instance_name",  "hire-right-lb",        "Lakebase instance name")
dbutils.widgets.text("lakebase_database",       "databricks_postgres",  "Lakebase Postgres database")
dbutils.widgets.text("lakebase_catalog",        "hrd_2030_lakebase",    "UC catalog for synced tables (no hyphens)")
dbutils.widgets.text("capacity",                "CU_1",                 "Instance capacity")

In [ ]:
import os, time, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
logger = logging.getLogger("lakebase-setup")

# Widget takes priority (job passes it); fall back to .env for interactive runs
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + _nb_path.rsplit("/notebooks", 1)[0]
    from dotenv import load_dotenv
    load_dotenv(f"{_root}/.env")
except Exception:
    pass

CATALOG           = dbutils.widgets.get("catalog")                or os.getenv("TARGET_CATALOG", "bldemos")
SCHEMA            = dbutils.widgets.get("schema")                 or os.getenv("TARGET_SCHEMA", "hrd_2030")
INSTANCE_NAME     = dbutils.widgets.get("lakebase_instance_name") or os.getenv("LAKEBASE_INSTANCE_NAME", "hire-right-lb")
LAKEBASE_DATABASE = dbutils.widgets.get("lakebase_database")      or os.getenv("LAKEBASE_DATABASE", "databricks_postgres")
LAKEBASE_CATALOG  = dbutils.widgets.get("lakebase_catalog")       or os.getenv("LAKEBASE_CATALOG", "hrd_2030_lakebase")
CAPACITY          = dbutils.widgets.get("capacity")              or "CU_1"

# Delta tables the app UI reads -> synced to Postgres. Array cols become JSONB.
TABLES_TO_SYNC = [
    {"table": "candidate_scoring_summary", "pk": ["candidate_id"], "policy": "TRIGGERED"},
    {"table": "job_requirements",          "pk": ["job_id"],       "policy": "TRIGGERED"},
]

# The native (non-synced) transactional table for HR annotations lives in the
# same Postgres schema as the synced tables, so it can join on candidate_id.
ANNOTATIONS_TABLE = "candidate_annotations"

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

print(f"Catalog        : {CATALOG}")
print(f"Schema         : {SCHEMA}")
print(f"Instance       : {INSTANCE_NAME}  ({CAPACITY})")
print(f"DB catalog     : {LAKEBASE_CATALOG} -> {INSTANCE_NAME}/{LAKEBASE_DATABASE}")
print(f"Tables to sync : {[t['table'] for t in TABLES_TO_SYNC]}")

## 1. Create (or reuse) the Lakebase instance

If it already exists we reuse it — creation is skipped. A stopped instance is started.

In [ ]:
from databricks.sdk.service.database import DatabaseInstance

def create_or_get_instance():
    """Return an AVAILABLE Lakebase instance, creating it only if missing."""
    try:
        inst = w.database.get_database_instance(name=INSTANCE_NAME)
        state = str(inst.state).upper() if inst.state else "UNKNOWN"
        logger.info(f"Instance '{INSTANCE_NAME}' already exists (state={state}) — skipping creation.")
        if "STOPPED" in state:
            logger.info("Starting stopped instance...")
            w.database.update_database_instance(
                name=INSTANCE_NAME,
                database_instance=DatabaseInstance(name=INSTANCE_NAME, stopped=False),
                update_mask="stopped",
            )
            for _ in range(60):
                time.sleep(5)
                inst = w.database.get_database_instance(name=INSTANCE_NAME)
                if inst.state and "AVAILABLE" in str(inst.state).upper():
                    break
        return w.database.get_database_instance(name=INSTANCE_NAME)
    except Exception as e:
        if "not found" not in str(e).lower() and "does not exist" not in str(e).lower() and "resource_does_not_exist" not in str(e).lower():
            logger.warning(f"get_database_instance raised (treating as missing): {e}")

    logger.info(f"Creating Lakebase instance '{INSTANCE_NAME}' ({CAPACITY})...")
    inst = w.database.create_database_instance_and_wait(
        DatabaseInstance(name=INSTANCE_NAME, capacity=CAPACITY)
    )
    logger.info(f"Instance ready. DNS: {inst.read_write_dns}")
    return inst

instance = create_or_get_instance()
PG_HOST = instance.read_write_dns
print(f"✓ Instance available. Host: {PG_HOST}")

## 2. Register the Postgres-backed database catalog

Create-if-missing; safe to re-run.

In [ ]:
from databricks.sdk.service.database import DatabaseCatalog

def ensure_database_catalog():
    try:
        w.database.create_database_catalog(
            DatabaseCatalog(
                name=LAKEBASE_CATALOG,
                database_instance_name=INSTANCE_NAME,
                database_name=LAKEBASE_DATABASE,
                create_database_if_not_exists=True,
            )
        )
        logger.info(f"Database catalog '{LAKEBASE_CATALOG}' -> {INSTANCE_NAME}/{LAKEBASE_DATABASE} created.")
    except Exception as e:
        if "already" in str(e).lower() or "exists" in str(e).lower():
            logger.info(f"Database catalog '{LAKEBASE_CATALOG}' already exists — skipping.")
        else:
            raise

ensure_database_catalog()

## 3. Sync the Delta tables → Postgres

TRIGGERED synced tables require Change Data Feed on the source Delta table.
Existing synced tables are left in place (skip-if-exists); additive schema
changes on the source propagate automatically on the next sync.

In [ ]:
from databricks.sdk.service.sql import StatementState

def _warehouse_id():
    wid = os.getenv("DATABRICKS_WAREHOUSE_ID")
    if wid:
        return wid
    for wh in w.warehouses.list():
        if wh.state and wh.state.value == "RUNNING":
            return wh.id
    for wh in w.warehouses.list():
        return wh.id
    raise RuntimeError("No SQL warehouse available to enable CDF.")

def enable_cdf(source_table):
    sql = f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
    try:
        resp = w.statement_execution.execute_statement(
            warehouse_id=_warehouse_id(), statement=sql, wait_timeout="30s"
        )
        if resp.status and resp.status.state == StatementState.FAILED:
            logger.warning(f"  CDF enable may have failed on {source_table}: {resp.status.error}")
        else:
            logger.info(f"  CDF enabled on {source_table}")
    except Exception as e:
        logger.warning(f"  Could not enable CDF on {source_table}: {e}")

In [ ]:
from databricks.sdk.service.database import (
    SyncedDatabaseTable, SyncedTableSpec, SyncedTableSchedulingPolicy, NewPipelineSpec,
)

policy_map = {
    "SNAPSHOT":   SyncedTableSchedulingPolicy.SNAPSHOT,
    "TRIGGERED":  SyncedTableSchedulingPolicy.TRIGGERED,
    "CONTINUOUS": SyncedTableSchedulingPolicy.CONTINUOUS,
}

def synced_table_exists(target_table):
    try:
        w.database.get_synced_database_table(name=target_table)
        return True
    except Exception:
        return False

for cfg in TABLES_TO_SYNC:
    name   = cfg["table"]
    source = f"{CATALOG}.{SCHEMA}.{name}"
    target = f"{LAKEBASE_CATALOG}.{SCHEMA}.{name}"
    policy = cfg["policy"]

    if policy in ("TRIGGERED", "CONTINUOUS"):
        enable_cdf(source)

    # Delete any existing synced table first, then recreate. The Gold source is
    # rebuilt by 03_build_gold on every pipeline run (new table identity / CDF),
    # so a skip-if-exists here would leave the sync pointing at a replaced source
    # and serve stale data. Recreating guarantees the sync matches the current source.
    if synced_table_exists(target):
        logger.info(f"Removing existing synced table so it can be recreated: {name}")
        try:
            w.database.delete_synced_database_table(name=target)
            time.sleep(3)
        except Exception as e:
            logger.warning(f"  Could not delete existing synced table {name}: {e}")

    logger.info(f"Creating synced table: {source} -> {target} ({policy})")
    try:
        w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=target,
                database_instance_name=INSTANCE_NAME,
                logical_database_name=LAKEBASE_DATABASE,
                spec=SyncedTableSpec(
                    source_table_full_name=source,
                    primary_key_columns=cfg["pk"],
                    scheduling_policy=policy_map[policy],
                    create_database_objects_if_missing=True,
                    # This metastore may have no storage root; point the backing
                    # pipeline at a catalog that has managed storage.
                    new_pipeline_spec=NewPipelineSpec(
                        storage_catalog=CATALOG,
                        storage_schema=SCHEMA,
                    ),
                ),
            )
        )
        logger.info(f"  ✓ Synced table created: {name}")
    except Exception as e:
        if "already exists" in str(e).lower():
            logger.info(f"  ✓ Synced table already exists: {name}")
        else:
            raise

### Wait for the initial sync to complete

In [ ]:
pending = {t["table"] for t in TABLES_TO_SYNC}
for attempt in range(60):
    still = set()
    for name in pending:
        full = f"{LAKEBASE_CATALOG}.{SCHEMA}.{name}"
        try:
            st = w.database.get_synced_database_table(name=full)
            ss = st.data_synchronization_status
            state = str(ss.detailed_state).upper() if ss and ss.detailed_state else ""
            if any(s in state for s in ("ACTIVE", "ONLINE", "SUCCEEDED")):
                logger.info(f"  ✓ {name} sync complete ({state})")
                continue
            still.add(name)
        except Exception:
            still.add(name)
    pending = still
    if not pending:
        logger.info("All tables synced.")
        break
    if attempt % 6 == 0:
        logger.info(f"  {len(pending)} still syncing: {', '.join(sorted(pending))}")
    time.sleep(10)
if pending:
    logger.warning(f"Some tables may still be syncing: {', '.join(sorted(pending))}")

## 4. Create the transactional `candidate_annotations` table

A native Postgres table (not synced) for HR notes. It lives in the same schema
as the synced tables so it can join to candidate info on `candidate_id`.
`CREATE ... IF NOT EXISTS` keeps this idempotent; grants to `PUBLIC` let the
app service principal read & write regardless of its Postgres role.

In [ ]:
import psycopg

def _pg_connect():
    username = w.config.client_id if w.config.client_id else w.current_user.me().user_name
    cred = w.database.generate_database_credential(instance_names=[INSTANCE_NAME])
    return psycopg.connect(
        host=PG_HOST, dbname=LAKEBASE_DATABASE, user=username,
        password=cred.token, sslmode="require", port=int(os.getenv("PGPORT", "5432")),
        autocommit=True,
    )

DDL = f"""
CREATE SCHEMA IF NOT EXISTS {SCHEMA};

CREATE TABLE IF NOT EXISTS {SCHEMA}.{ANNOTATIONS_TABLE} (
    id           BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    candidate_id TEXT        NOT NULL,
    note         TEXT        NOT NULL,
    author       TEXT,
    created_at   TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE INDEX IF NOT EXISTS idx_{ANNOTATIONS_TABLE}_candidate
    ON {SCHEMA}.{ANNOTATIONS_TABLE} (candidate_id);

COMMENT ON TABLE {SCHEMA}.{ANNOTATIONS_TABLE} IS
    'Transactional HR annotations/notes about candidates. Join to candidate_scoring_summary on candidate_id.';

GRANT USAGE ON SCHEMA {SCHEMA} TO PUBLIC;
GRANT SELECT, INSERT, UPDATE, DELETE ON {SCHEMA}.{ANNOTATIONS_TABLE} TO PUBLIC;
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA {SCHEMA} TO PUBLIC;
-- Let the app service principal read the synced tables too (they are
-- owned by the sync pipeline, so grant SELECT on everything in the schema).
GRANT SELECT ON ALL TABLES IN SCHEMA {SCHEMA} TO PUBLIC;
"""

with _pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL)
        cur.execute(f"SELECT count(*) FROM {SCHEMA}.{ANNOTATIONS_TABLE}")
        n = cur.fetchone()[0]
print(f"✓ {SCHEMA}.{ANNOTATIONS_TABLE} ready ({n} existing rows)")

### Smoke test: join annotations to candidate info

In [ ]:
with _pg_connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"SET search_path TO {SCHEMA}, public")
        cur.execute(f"""
            SELECT c.candidate_id, c.full_name, c.job_title, c.total_score, c.stage,
                   count(a.id) AS note_count
            FROM candidate_scoring_summary c
            LEFT JOIN {ANNOTATIONS_TABLE} a ON a.candidate_id = c.candidate_id
            GROUP BY 1,2,3,4,5
            ORDER BY c.total_score DESC NULLS LAST
            LIMIT 10
        """)
        for row in cur.fetchall():
            print(row)

## 5. Summary — env vars for the app

In [ ]:
print("=" * 60)
print("Lakebase setup complete")
print("=" * 60)
print(f"LAKEBASE_INSTANCE_NAME = {INSTANCE_NAME}")
print(f"LAKEBASE_DATABASE      = {LAKEBASE_DATABASE}")
print(f"LAKEBASE_HOST          = {PG_HOST}")
print(f"PG_SCHEMA              = {SCHEMA}")
print(f"Synced tables          = {LAKEBASE_DATABASE}.{SCHEMA}.{{candidate_scoring_summary, job_requirements}}")
print(f"Annotations table      = {LAKEBASE_DATABASE}.{SCHEMA}.{ANNOTATIONS_TABLE}")

# Surface the host to the job so downstream tasks (deploy_app) can consume it.
try:
    dbutils.jobs.taskValues.set(key="lakebase_host", value=PG_HOST)
    dbutils.jobs.taskValues.set(key="lakebase_instance_name", value=INSTANCE_NAME)
except Exception:
    pass